In [1]:
from datetime import datetime, timedelta, timezone

from seismic_intelligence.ingestion.usgs_client import (
    USGSEarthquakeClient
)

from seismic_intelligence.ingestion.geojson_parser import (
    parse_geojson
)

from seismic_intelligence.transformations.earthquake_transformer import (
    transform_earthquake_events
)

In [2]:
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=3)

with USGSEarthquakeClient() as client:

    payload = client.get_earthquakes(
        starttime=start_time,
        endtime=end_time,
        min_magnitude=2.5,
        limit=100
    )

events = parse_geojson(payload)

df_earthquakes = transform_earthquake_events(events)

print("Extracted:", len(events))
print("Transformed:", len(df_earthquakes))

display(df_earthquakes.head(10))

Extracted: 100
Transformed: 100


,event_id,magnitude,magnitude_type,place,event_time_utc,updated_time_utc,longitude,latitude,depth_km,event_type,status,tsunami,alert,significance,event_url
0,us7000ttb5,4.5,mb,"74 km SE of Rasht, Tajikistan",2026-10-26 18:07:04.209000+00:00,2026-10-26 18:29:32.040000+00:00,71.076400,38.644200,47.3450,earthquake,reviewed,0,<NA>,312,https://earthquake.usgs.gov/earthquakes/eventp...
1,aka2026vfmqfn,2.8,ml,"88 km ENE of McCarthy, Alaska",2026-10-26 14:02:54.892000+00:00,2026-10-26 14:06:02.846000+00:00,-141.316000,61.636000,5.0000,earthquake,automatic,0,<NA>,121,https://earthquake.usgs.gov/earthquakes/eventp...
2,nc75455061,2.5,md,"4 km SE of Aromas, CA",2026-10-26 13:10:33.190000+00:00,2026-10-26 16:43:00.468000+00:00,-121.605003,36.862667,4.8300,earthquake,automatic,0,<NA>,98,https://earthquake.usgs.gov/earthquakes/eventp...
3,us6000u4zb,5.2,mww,"205 km NW of Port-Olry, Vanuatu",2026-10-26 12:13:30.652000+00:00,2026-10-26 12:30:59.040000+00:00,165.602900,-13.861300,10.0000,earthquake,reviewed,0,<NA>,416,https://earthquake.usgs.gov/earthquakes/eventp...
4,us6000u4z7,4.6,mwr,"81 km SSE of Akutan, Alaska",2026-10-26 11:38:06.778000+00:00,2026-10-26 11:48:30.831000+00:00,-165.123900,53.507100,35.0000,earthquake,reviewed,0,<NA>,326,https://earthquake.usgs.gov/earthquakes/eventp...
5,us6000u4z6,5.0,mww,"218 km NW of Port-Olry, Vanuatu",2026-10-26 11:17:55.055000+00:00,2026-10-26 11:34:36.040000+00:00,165.385800,-13.939600,10.0000,earthquake,reviewed,0,<NA>,385,https://earthquake.usgs.gov/earthquakes/eventp...
6,us6000u4z3,4.8,mb,South Sandwich Islands region,2026-10-26 11:01:55.933000+00:00,2026-10-26 15:39:30.560000+00:00,-27.708700,-59.814100,173.4930,earthquake,reviewed,0,<NA>,354,https://earthquake.usgs.gov/earthquakes/eventp...
7,us6000u4yw,4.3,mb,"287 km E of Levuka, Fiji",2026-10-26 10:30:22.297000+00:00,2026-10-26 10:44:44.040000+00:00,-177.980100,-18.318700,636.6210,earthquake,reviewed,0,<NA>,284,https://earthquake.usgs.gov/earthquakes/eventp...
8,us6000u4yr,5.5,mww,"283 km E of Levuka, Fiji",2026-10-26 10:13:01.234000+00:00,2026-10-26 12:19:10.404000+00:00,-178.010900,-18.283000,641.8120,earthquake,reviewed,0,green,465,https://earthquake.usgs.gov/earthquakes/eventp...
9,tx2026vbupcx,2.6,ml,"58 km S of Whites City, New Mexico",2026-10-26 10:09:03.981000+00:00,2026-10-26 14:33:35.851000+00:00,-104.396000,31.652000,6.4587,earthquake,reviewed,0,<NA>,104,https://earthquake.usgs.gov/earthquakes/eventp...


In [4]:

from seismic_intelligence.quality.profiling import (
    profile_earthquake_data
)

quality_report = profile_earthquake_data(df_earthquakes)

# Summary metrics
print("========== QUALITY SUMMARY ==========")

for metric, value in quality_report["summary"].items():
    print(f"{metric}: {value}")

# Null profiling
print("\n========== NULL PROFILING ==========")
display(quality_report["null_profile"])

# Duplicate records
print("\n========== DUPLICATE RECORDS ==========")
display(quality_report["duplicate_records"])

# Invalid coordinates
print("\n========== INVALID COORDINATES ==========")
display(quality_report["invalid_coordinates"])


========== QUALITY SUMMARY ==========
total_records: 100
total_columns: 15
unique_event_ids: 100
duplicate_event_ids: 0
missing_event_ids: 0
missing_magnitudes: 0
missing_event_dates: 0
missing_coordinates: 0
invalid_latitudes: 0
invalid_longitudes: 0
invalid_coordinate_records: 0

========== NULL PROFILING ==========


,column_name,null_count,null_percentage
12,alert,93,93.0
1,magnitude,0,0.0
0,event_id,0,0.0
3,place,0,0.0
4,event_time_utc,0,0.0
5,updated_time_utc,0,0.0
2,magnitude_type,0,0.0
6,longitude,0,0.0
7,latitude,0,0.0
9,event_type,0,0.0



========== DUPLICATE RECORDS ==========


,event_id,magnitude,magnitude_type,place,event_time_utc,updated_time_utc,longitude,latitude,depth_km,event_type,status,tsunami,alert,significance,event_url



========== INVALID COORDINATES ==========


,event_id,magnitude,magnitude_type,place,event_time_utc,updated_time_utc,longitude,latitude,depth_km,event_type,status,tsunami,alert,significance,event_url


In [5]:
print(df_earthquakes.dtypes)

event_id                 string[python]
magnitude                       float64
magnitude_type           string[python]
place                    string[python]
event_time_utc      datetime64[ns, UTC]
updated_time_utc    datetime64[ns, UTC]
longitude                       float64
latitude                        float64
depth_km                        float64
event_type               string[python]
status                   string[python]
tsunami                           int64
alert                    string[python]
significance                      int64
event_url                string[python]
dtype: object
